# WoodelfHD — Depth Sweep (Notebook 1 / 3)

Runs **WoodelfHD** (`woodelf_for_high_depth`) across all datasets and task types
in the `woodelfhd_depth_sweep_experiment`.  
Notebooks 02 and 03 run OriginalWoodelf and SHAP **in parallel** on separate sessions.
Notebook 04 merges all three result files and generates the HTML report.

### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Clones `treebranchmarks` and `woodelf_explainer` repos
3. Installs all dependencies
4. Runs `woodelfhd_depth_sweep_experiment --method woodelf_hd`
5. Writes partial results to Drive as `partial_woodelf_hd.json`

### Datasets (all download automatically)
| Dataset | Source |
|---------|--------|
| Fraud Detection | Google Drive parquet (~200 MB) |
| HIGGS | UCI direct download (~8 GB uncompressed — takes ~20 min) |
| KDD Cup (Intrusion Detection) | Google Drive parquet |
| California Housing | sklearn builtin |

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
# Edit DRIVE_FOLDER to point to your preferred Google Drive folder.
# All other paths are derived from it.
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_sweep')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

DRIVE_RESULT_PATH = DRIVE_FOLDER / 'partial_woodelf_hd.json'
print(f'Results will be saved to: {DRIVE_RESULT_PATH}')

In [ ]:
# ── Step 3: Clone repositories ───────────────────────────────────────────────
# Replace the two URLs below with the actual repository URLs.

TREEBRANCHMARKS_URL = 'YOUR_TREEBRANCHMARKS_REPO_URL'  # e.g. https://github.com/you/treebranchmarks.git
WOODELF_URL         = 'YOUR_WOODELF_REPO_URL'          # e.g. https://github.com/you/woodelf_explainer.git

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks
!git clone {WOODELF_URL}         /content/woodelf_explainer

In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
# woodelf_explainer must be installed before treebranchmarks (it is listed
# as a dependency in treebranchmarks/pyproject.toml).

!pip install -q -e /content/woodelf_explainer
!pip install -q -e /content/treebranchmarks

In [ ]:
# ── Step 5: Run the experiment (WoodelfHD only) ───────────────────────────────
# -u   : unbuffered output so progress prints appear in real time
# --method woodelf_hd : only the WoodelfHD approach is timed
# --result_location   : dual-write to Drive after every completed mission

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method woodelf_hd \
    --result_location "{DRIVE_RESULT_PATH}"

In [ ]:
# ── Step 6: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    data = json.load(f)

n_missions    = len(data['missions'])
n_task_results = sum(len(m['task_results']) for m in data['missions'])

# Collect all approach names that were recorded
approach_names = set()
for m in data['missions']:
    for tr in m['task_results']:
        approach_names.update(tr['approach_results'].keys())

print(f'Missions:      {n_missions}')
print(f'Task results:  {n_task_results}')
print(f'Approaches:    {sorted(approach_names)}')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')